In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251206_072956.csv
Loaded: NBA_DFS_20251206_073400.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Michael Porter Jr,Over,25.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
1,PrizePicks,player_points,Michael Porter Jr,Under,25.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
2,PrizePicks,player_points,Trey Murphy III,Over,20.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
3,PrizePicks,player_points,Trey Murphy III,Under,20.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
4,PrizePicks,player_points,Saddiq Bey,Over,17.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00


In [10]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

# Test prediction for a single player
result = engine.project_player(
    player_name="Ivica Zubac",
    data=s26,
    date="2025-12-06",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

print("Prediction Result:")
print(f"  Predicted Minutes: {result['predicted_minutes']:.2f}")
print(f"  Predicted Usage: {result['predicted_usage']:.3f}")
print(f"  Predicted FGA: {result['predicted_fga']:.2f}")
print(f"  Predicted Points: {result['predicted_points']:.2f}")

# Check if model calibrations are loaded
if engine.ngboost_model_wrapper and 'model_calibrations' in engine.ngboost_model_wrapper:
    calibrations = engine.ngboost_model_wrapper['model_calibrations']
    if 'PTS' in calibrations:
        print(f"  Variance Calibration (PTS): {calibrations['PTS']}")

Prediction Result:
  Predicted Minutes: 31.76
  Predicted Usage: 0.173
  Predicted FGA: 10.26
  Predicted Points: 17.94
  Variance Calibration (PTS): {'low': 0.8, 'medium': 1.2, 'high': 1.8}


In [4]:
# Filter for Underdog player points
# dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]


really_low_volume_lines = dfsPTS[dfsPTS['LINE'].between(4.5, 8.5)]
print(f"Really low volume lines (4.5-7.5): {len(really_low_volume_lines)} bets")

low_volume_lines = dfsPTS[dfsPTS['LINE'].between(8.5, 15.5)]
print(f"Low volume lines (8.5-15.5): {len(low_volume_lines)} bets")

mid_tier_lines = dfsPTS[dfsPTS['LINE'].between(15.5, 22.5)]
print(f"Mid-tier lines (15.5-22.5): {len(mid_tier_lines)} bets")

high_volume_lines = dfsPTS[dfsPTS['LINE'].between(22.5, 35.5)]
print(f"High volume lines (22.5-35.5): {len(high_volume_lines)} bets")

ceiling_lines = dfsPTS[dfsPTS['LINE'] >= 30.5]
print(f"Ceiling lines (30.5+): {len(ceiling_lines)} bets")

Really low volume lines (4.5-7.5): 42 bets
Low volume lines (8.5-15.5): 96 bets
Mid-tier lines (15.5-22.5): 48 bets
High volume lines (22.5-35.5): 22 bets
Ceiling lines (30.5+): 0 bets


In [5]:
underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=really_low_volume_lines,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
underdogPairs

Computing predictions for 20 players...
Found 20 valid players
Generated 167 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,TOTAL_EDGE
55,Micah Peavy,Isaiah Stewart,4.5,7.5,over,over,4.58,7.64,0.08,0.14,0.22
71,Vít Krejčí,Mike Conley,8.5,4.5,under,under,8.31,4.33,-0.19,-0.17,-0.36
153,Kobe Sanders,Dru Smith,7.0,6.0,under,under,6.70,5.64,-0.30,-0.36,-0.66
24,Jordan Hawkins,Kris Dunn,7.0,7.5,under,under,6.22,6.47,-0.78,-1.03,-1.81
108,Mouhamed Gueye,Precious Achiuwa,6.5,7.5,under,under,4.99,6.07,-1.51,-1.43,-2.94
98,Luke Kennard,Drew Eubanks,7.5,4.5,under,under,5.90,2.98,-1.60,-1.52,-3.12
5,Terance Mann,Dean Wade,8.5,5.5,under,under,6.79,3.64,-1.71,-1.86,-3.57
159,Nicolas Batum,Josh Okogie,4.5,6.5,under,under,2.41,4.35,-2.09,-2.15,-4.24
33,Yves Missi,Bub Carrington,6.5,8.0,under,under,4.21,5.63,-2.29,-2.37,-4.66
112,Quinten Post,Caris LeVert,8.5,8.0,under,under,4.84,5.42,-3.66,-2.58,-6.24


In [6]:
underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=low_volume_lines,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
underdogPairs

Computing predictions for 48 players...
[MIN] No data found for Egor Demin
Found 47 valid players
Generated 945 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,TOTAL_EDGE
871,Jaden McDaniels,Jaime Jaquez Jr.,12.5,13.5,over,over,19.39,22.29,6.89,8.79,15.68
834,Ivica Zubac,P.J. Washington,14.5,13.5,over,over,17.94,17.49,3.44,3.99,7.43
398,Zaccharie Risacher,Naz Reid,11.5,13.5,over,over,13.70,16.63,2.20,3.13,5.33
197,Ziaire Williams,Jabari Smith Jr.,9.5,13.5,over,over,10.06,15.33,0.56,1.83,2.39
769,Ausar Thompson,Reed Sheppard,10.5,10.5,over,over,10.60,12.13,0.10,1.63,1.73
343,Dyson Daniels,Klay Thompson,12.5,11.5,under,over,12.40,12.44,-0.10,0.94,0.84
475,Vít Krejčí,Davion Mitchell,8.5,9.5,under,under,8.31,9.37,-0.19,-0.13,-0.32
153,Jose Alvarado,Kel'el Ware,9.5,12.5,under,under,9.16,12.17,-0.34,-0.33,-0.67
597,Jaylon Tyson,Duncan Robinson,12.5,10.5,under,under,11.93,10.14,-0.57,-0.36,-0.93
61,Nic Claxton,Gary Trent Jr.,14.5,9.5,under,under,13.63,8.85,-0.87,-0.65,-1.52


In [7]:
underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=mid_tier_lines,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
underdogPairs

Computing predictions for 24 players...
Found 24 valid players
Generated 242 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,TOTAL_EDGE
86,Nickeil Alexander-Walker,Julius Randle,20.5,20.5,over,over,25.30,26.11,4.80,5.61,10.41
15,Trey Murphy III,Andrew Wiggins,20.5,16.5,over,over,25.03,20.63,4.53,4.13,8.66
51,Jeremiah Fears,Bam Adebayo,16.5,20.5,over,over,17.06,24.49,0.56,3.99,4.55
237,Keegan Murray,Cooper Flagg,17.5,16.5,over,over,18.97,16.82,1.47,0.32,1.79
124,Onyeka Okongwu,Alperen Sengun,18.5,22.5,under,under,18.18,22.18,-0.32,-0.32,-0.64
232,DeMar DeRozan,Amen Thompson,18.5,17.5,under,under,17.98,16.89,-0.52,-0.61,-1.13
162,Evan Mobley,Zach LaVine,18.5,19.5,under,under,16.66,17.94,-1.84,-1.56,-3.40
98,CJ McCollum,Darius Garland,19.5,16.5,under,under,17.50,12.62,-2.00,-3.88,-5.88
37,Saddiq Bey,Anthony Davis,17.5,19.5,under,under,13.26,15.56,-4.24,-3.94,-8.18
132,Kyshawn George,Ryan Rollins,15.5,20.5,under,under,10.77,15.91,-4.73,-4.59,-9.32


In [8]:
underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=high_volume_lines,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
underdogPairs

Computing predictions for 11 players...
Found 11 valid players
Generated 50 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,TOTAL_EDGE
7,Michael Porter Jr.,Tyler Herro,25.5,23.5,over,over,27.95,30.00,2.45,6.50,8.95
42,Kawhi Leonard,Norman Powell,23.5,24.5,over,over,23.83,25.31,0.33,0.81,1.14
37,Anthony Edwards,Alperen Sengun,29.5,22.5,over,under,29.61,22.18,0.11,-0.32,-0.21
19,Donovan Mitchell,Cade Cunningham,29.5,27.5,under,under,27.92,25.88,-1.58,-1.62,-3.20
13,Jalen Johnson,James Harden,25.5,25.5,under,under,23.82,22.60,-1.68,-2.90,-4.58


In [9]:
underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=ceiling_lines,
    engine=engine,
    current_date=current_date,
    top_n=10,                   
    max_player_appearances=1,   
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
underdogPairs

No bets found for player_points


""
